In [1]:
#Data Manipulation specific
import numpy as np
import pandas as pd
import xarray as xr
#import scipy.stats as stats
#from scipy import signal
from netCDF4 import Dataset
 
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap

import rasterio
from rasterio.transform import from_origin

import os

/home/anas.oubaha/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
!free -g

              total        used        free      shared  buff/cache   available
Mem:            376          60         134           1         180         311
Swap:             0           0           0


**Read NC data and extract u and v wind**

In [4]:
# Load dataset
dw = xr.open_dataset('data/850&200PH_Data_80N20S50W80E.nc', engine="netcdf4")  # Load NetCDF file
#dw = dw.sel(latitude=slice(45, 15))
#dw = dw.sel(valid_time=slice('1991-01-01', None))
dw = dw[['u', 'v']] #keep only the variables u and v
dwl = dw.where(dw.pressure_level.isin([850]), drop=True)
dwl = dwl.squeeze()
dwh = dw.where(dw.pressure_level.isin([200]), drop=True)
dwh = dwh.squeeze()

In [5]:
dwl

<xarray.Dataset>
Dimensions:         (valid_time: 408, latitude: 401, longitude: 521)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 1991-01-01 ... 2024-12-01
    pressure_level  float64 850.0
  * latitude        (latitude) float64 80.0 79.75 79.5 ... -19.5 -19.75 -20.0
  * longitude       (longitude) float64 -50.0 -49.75 -49.5 ... 79.5 79.75 80.0
Data variables:
    u               (valid_time, latitude, longitude) float32 0.8316 ... -7.368
    v               (valid_time, latitude, longitude) float32 7.593 ... 2.036
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

**Seasonal GPH and wind anomalies**

In [7]:
# Convert 'date' from integer to datetime format
dwh = dwh.assign_coords(date=pd.to_datetime(dwh['valid_time'].data.astype(str), format='ISO8601'))

# Convert dataset to DataFrame
dfu = dwh['u'].to_dataframe().reset_index()
dfv = dwh['v'].to_dataframe().reset_index()

# Extract year and month
dfu['year'] = dfu['valid_time'].dt.year
dfu['month'] = dfu['valid_time'].dt.month
dfv['year'] = dfv['valid_time'].dt.year
dfv['month'] = dfv['valid_time'].dt.month

# **FAST NumPy Vectorized Method for Assigning Seasons**
seasons = np.array(["DJF", "MAM", "JJA", "SON"])

# Define conditions for season assignment
winter = dfu['month'].isin([12, 1, 2])
spring = dfu['month'].isin([3, 4, 5])
summer = dfu['month'].isin([6, 7, 8])
autumn = dfu['month'].isin([9, 10, 11])

# Assign seasons using np.select (much faster than apply)
dfu['season'] = np.select([winter, spring, summer, autumn], seasons)
dfv['season'] = np.select([winter, spring, summer, autumn], seasons)

# Assign season_year (December belongs to next year’s winter)
dfu['season_year'] = dfu['year']
dfu.loc[dfu['month'] == 12, 'season_year'] += 1  # Shift December to next year's winter
dfv['season_year'] = dfv['year']
dfv.loc[dfv['month'] == 12, 'season_year'] += 1

# **STEP 1: Compute Actual Seasonal SST (For Each Year)**
seasonal_u_df = dfu.groupby(['latitude', 'longitude', 'season_year', 'season'])['u'].mean().reset_index()
seasonal_v_df = dfv.groupby(['latitude', 'longitude', 'season_year', 'season'])['v'].mean().reset_index()

# **STEP 2: Compute Seasonal Climatology (Long-Term Average Over All Years
# Filter to only include seasons from 1991 to 2020 (30 years period recommended by the WMO)
filtered_dfu = seasonal_u_df[(seasonal_u_df['season_year'] >= 1991) & (seasonal_u_df['season_year'] <= 2020)]
filtered_dfv = seasonal_v_df[(seasonal_v_df['season_year'] >= 1991) & (seasonal_v_df['season_year'] <= 2020)]

seasonal_climatology_u = filtered_dfu.groupby(['latitude', 'longitude', 'season'])['u'].mean().reset_index()
seasonal_climatology_u.rename(columns={'u': 'u_climatology'}, inplace=True)
seasonal_climatology_v = filtered_dfv.groupby(['latitude', 'longitude', 'season'])['v'].mean().reset_index()
seasonal_climatology_v.rename(columns={'v': 'v_climatology'}, inplace=True)

# **STEP 3: Merge Actual SST With Climatology**
mergedfu = pd.merge(seasonal_u_df, seasonal_climatology_u, on=['latitude', 'longitude', 'season'], how="left")
mergedfv = pd.merge(seasonal_v_df, seasonal_climatology_v, on=['latitude', 'longitude', 'season'], how="left")

# **STEP 4: Compute Seasonal Anomalies**
mergedfu['u_anomaly'] = mergedfu['u'] - mergedfu['u_climatology']

mergedfu['date'] = (
    mergedfu['season'].astype(str) 
    + ' ' 
    + mergedfu['season_year'].astype(str)
)

mergedfv['v_anomaly'] = mergedfv['v'] - mergedfv['v_climatology']

mergedfv['date'] = (
    mergedfv['season'].astype(str) 
    + ' ' 
    + mergedfv['season_year'].astype(str)
)
mergedfu

,latitude,longitude,season_year,season,u,u_climatology,u_anomaly,date
0,-20.0,-50.0,1991,DJF,5.471332,4.463484,1.007848,DJF 1991
1,-20.0,-50.0,1991,JJA,24.721445,25.563805,-0.842360,JJA 1991
2,-20.0,-50.0,1991,MAM,15.276420,18.128437,-2.852017,MAM 1991
3,-20.0,-50.0,1991,SON,18.753584,18.235901,0.517683,SON 1991
4,-20.0,-50.0,1992,DJF,4.615769,4.463484,0.152285,DJF 1992
...,...,...,...,...,...,...,...,...
28622172,80.0,80.0,2024,DJF,0.182895,1.448011,-1.265116,DJF 2024
28622173,80.0,80.0,2024,JJA,5.163803,4.959941,0.203862,JJA 2024
28622174,80.0,80.0,2024,MAM,-1.671639,1.184545,-2.856184,MAM 2024
28622175,80.0,80.0,2024,SON,3.820954,3.266761,0.554193,SON 2024


**Apply Welch’s T-Test**

In [8]:
from scipy.stats import ttest_ind

# Define drought years
drought_years = [1994, 1995, 1999, 2000, 2001, 2005, 2016, 2019, 2020, 2021, 2022, 2023]  # Replace with your own

# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedfu.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='u_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge with original mergedfu
mergedfu = pd.merge(mergedfu, p_df, on=['latitude', 'longitude', 'season'], how='left')
##########################################################################################################
# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedfv.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='v_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge with original mergedfv
mergedfv = pd.merge(mergedfv, p_df, on=['latitude', 'longitude', 'season'], how='left')

############################################################################################
# Convert back to xarray
uanomalies = mergedfu.set_index(['date', 'latitude', 'longitude'])['u_anomaly'].to_xarray()
upval = mergedfu.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()

vanomalies = mergedfv.set_index(['date', 'latitude', 'longitude'])['v_anomaly'].to_xarray()
vpval = mergedfv.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()
uanomalies

<xarray.DataArray 'u_anomaly' (date: 137, latitude: 401, longitude: 521)>
array([[[ 1.0078478 ,  1.0642495 ,  1.098299  , ..., -5.1274495 ,
         -5.2341332 , -5.3372474 ],
        [ 0.94757223,  0.9796686 ,  0.9941759 , ..., -5.196796  ,
         -5.308006  , -5.405108  ],
        [ 0.88791513,  0.9047661 ,  0.90830326, ..., -5.241751  ,
         -5.3590903 , -5.449347  ],
        ...,
        [ 0.9712374 ,  1.0001438 ,  1.0279758 , ..., -1.4856093 ,
         -1.5055203 , -1.5268095 ],
        [ 0.97616386,  1.0042236 ,  1.0333793 , ..., -1.3530246 ,
         -1.373565  , -1.3932265 ],
        [ 0.9704671 ,  0.9989612 ,  1.0274334 , ..., -1.2226644 ,
         -1.2421956 , -1.2618355 ]],

       [[ 0.15228462,  0.19045734,  0.21669436, ...,  1.855854  ,
          1.9086752 ,  1.9816685 ],
        [ 0.2830901 ,  0.32104588,  0.34434223, ...,  1.9352703 ,
          1.9975634 ,  2.0902395 ],
        [ 0.4089799 ,  0.44178104,  0.4635477 , ...,  2.0215006 ,
          2.0968695 ,  2.2097378 ],
...
        [ 1.4127779 ,  1.4023938 ,  1.3918142 , ...,  1.9108896 ,
          1.9257877 ,  1.9412282 ],
        [ 1.4455361 ,  1.4358354 ,  1.4266663 , ...,  1.8378537 ,
          1.8518836 ,  1.8661194 ],
        [ 1.4587951 ,  1.4506569 ,  1.442584  , ...,  1.7673783 ,
          1.7798672 ,  1.7921722 ]],

       [[-3.7430725 , -3.6656542 , -3.585185  , ..., -3.9107056 ,
         -3.8940716 , -3.8864765 ],
        [-3.7006025 , -3.6241932 , -3.5471315 , ..., -4.0484543 ,
         -4.038082  , -4.0406857 ],
        [-3.6574278 , -3.5846415 , -3.5127888 , ..., -4.185912  ,
         -4.1866274 , -4.2003536 ],
        ...,
        [ 1.560215  ,  1.5348573 ,  1.5086527 , ...,  0.7829366 ,
          0.75551724,  0.7299416 ],
        [ 1.5063848 ,  1.4810591 ,  1.4549632 , ...,  0.6897185 ,
          0.66468596,  0.63985944],
        [ 1.442821  ,  1.415803  ,  1.3881989 , ...,  0.6010139 ,
          0.5770447 ,  0.554193  ]]], dtype=float32)
Coordinates:
  * date       (date) object 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 -20.0 -19.75 -19.5 -19.25 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 -50.0 -49.75 -49.5 -49.25 ... 79.5 79.75 80.0

In [9]:
def export_to_geotiff(path, u_data, v_data, upval, vpval, lat, lon, description):
    # Flip latitude if needed (GeoTIFF expects top-to-bottom order)
    if lat[0] > lat[-1]:
        u_data = u_data[::-1]
        v_data = v_data[::-1]
        upval = u_data[::-1]
        vpval = v_data[::-1]
        lat = lat[::-1]

    # Get transform
    pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1)
    pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1)
    transform = from_origin(lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y)

    # Stack u and v into a 3D array (bands, height, width)
    data = np.stack([u_data, v_data, upval, vpval])  # shape: (2, lat, lon)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=u_data.shape[0],
        width=u_data.shape[1],
        count=4,  # 2 bands: u and v
        dtype=u_data.dtype,
        crs="EPSG:4326",
        transform=transform,
        description=description
    ) as dst:
        dst.write(data[0], 1)  # Band 1 - u
        dst.write(data[1], 2) # Band 2 - v
        dst.write(data[2], 3)  # Band 3 - upval
        dst.write(data[3], 4) # Band 4 - vpval
        
# Export regression slope map
output_dir = 'Wind200'  # no leading slash unless it's an absolute path
os.makedirs(output_dir, exist_ok=True)

for d in range(137):
    file_path = os.path.join(output_dir, str(uanomalies['date'][d].values) + '.tif')
    export_to_geotiff(file_path, uanomalies[d], vanomalies[d], upval[d], vpval[d], uanomalies.latitude.values, uanomalies.longitude.values, 'Wind200 Seasonal Anomalies of '+str(uanomalies['date'][d].values))


In [11]:
# Convert 'date' from integer to datetime format
dwl = dwl.assign_coords(date=pd.to_datetime(dwl['valid_time'].data.astype(str), format='ISO8601'))

# Convert dataset to DataFrame
dfu = dwl['u'].to_dataframe().reset_index()
dfv = dwl['v'].to_dataframe().reset_index()

# Extract year and month
dfu['year'] = dfu['valid_time'].dt.year
dfu['month'] = dfu['valid_time'].dt.month
dfv['year'] = dfv['valid_time'].dt.year
dfv['month'] = dfv['valid_time'].dt.month

# **FAST NumPy Vectorized Method for Assigning Seasons**
seasons = np.array(["DJF", "MAM", "JJA", "SON"])

# Define conditions for season assignment
winter = dfu['month'].isin([12, 1, 2])
spring = dfu['month'].isin([3, 4, 5])
summer = dfu['month'].isin([6, 7, 8])
autumn = dfu['month'].isin([9, 10, 11])

# Assign seasons using np.select (much faster than apply)
dfu['season'] = np.select([winter, spring, summer, autumn], seasons)
dfv['season'] = np.select([winter, spring, summer, autumn], seasons)

# Assign season_year (December belongs to next year’s winter)
dfu['season_year'] = dfu['year']
dfu.loc[dfu['month'] == 12, 'season_year'] += 1  # Shift December to next year's winter
dfv['season_year'] = dfv['year']
dfv.loc[dfv['month'] == 12, 'season_year'] += 1

# **STEP 1: Compute Actual Seasonal SST (For Each Year)**
seasonal_u_df = dfu.groupby(['latitude', 'longitude', 'season_year', 'season'])['u'].mean().reset_index()
seasonal_v_df = dfv.groupby(['latitude', 'longitude', 'season_year', 'season'])['v'].mean().reset_index()

# **STEP 2: Compute Seasonal Climatology (Long-Term Average Over All Years
# Filter to only include seasons from 1991 to 2020 (30 years period recommended by the WMO)
filtered_dfu = seasonal_u_df[(seasonal_u_df['season_year'] >= 1991) & (seasonal_u_df['season_year'] <= 2020)]
filtered_dfv = seasonal_v_df[(seasonal_v_df['season_year'] >= 1991) & (seasonal_v_df['season_year'] <= 2020)]

seasonal_climatology_u = filtered_dfu.groupby(['latitude', 'longitude', 'season'])['u'].mean().reset_index()
seasonal_climatology_u.rename(columns={'u': 'u_climatology'}, inplace=True)
seasonal_climatology_v = filtered_dfv.groupby(['latitude', 'longitude', 'season'])['v'].mean().reset_index()
seasonal_climatology_v.rename(columns={'v': 'v_climatology'}, inplace=True)

# **STEP 3: Merge Actual SST With Climatology**
mergedfu = pd.merge(seasonal_u_df, seasonal_climatology_u, on=['latitude', 'longitude', 'season'], how="left")
mergedfv = pd.merge(seasonal_v_df, seasonal_climatology_v, on=['latitude', 'longitude', 'season'], how="left")

# **STEP 4: Compute Seasonal Anomalies**
mergedfu['u_anomaly'] = mergedfu['u'] - mergedfu['u_climatology']

mergedfu['date'] = (
    mergedfu['season'].astype(str) 
    + ' ' 
    + mergedfu['season_year'].astype(str)
)

mergedfv['v_anomaly'] = mergedfv['v'] - mergedfv['v_climatology']

mergedfv['date'] = (
    mergedfv['season'].astype(str) 
    + ' ' 
    + mergedfv['season_year'].astype(str)
)
mergedfu

,latitude,longitude,season_year,season,u,u_climatology,u_anomaly,date
0,-20.0,-50.0,1991,DJF,1.265694,0.148495,1.117199,DJF 1991
1,-20.0,-50.0,1991,JJA,-1.967743,-1.096187,-0.871556,JJA 1991
2,-20.0,-50.0,1991,MAM,-0.964238,-1.086227,0.121989,MAM 1991
3,-20.0,-50.0,1991,SON,-0.973602,-0.884499,-0.089104,SON 1991
4,-20.0,-50.0,1992,DJF,0.261227,0.148495,0.112733,DJF 1992
...,...,...,...,...,...,...,...,...
28622172,80.0,80.0,2024,DJF,0.275273,-0.248256,0.523528,DJF 2024
28622173,80.0,80.0,2024,JJA,2.952901,1.279713,1.673188,JJA 2024
28622174,80.0,80.0,2024,MAM,-1.844981,-0.894474,-0.950507,MAM 2024
28622175,80.0,80.0,2024,SON,-1.286657,0.424442,-1.711099,SON 2024


In [12]:
from scipy.stats import ttest_ind

# Define drought years
drought_years = [1994, 1995, 1999, 2000, 2001, 2005, 2016, 2019, 2020, 2021, 2022, 2023]  # Replace with your own

# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedfu.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='u_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge with original mergedfu
mergedfu = pd.merge(mergedfu, p_df, on=['latitude', 'longitude', 'season'], how='left')
##########################################################################################################
# Create a multi-index pivot: rows = (season_year), columns = (lat, lon, season), values = anomaly
pivot = mergedfv.pivot_table(
    index='season_year',
    columns=['latitude', 'longitude', 'season'],
    values='v_anomaly'
)

# Get drought and non-drought subsets
drought_vals = pivot.loc[pivot.index.isin(drought_years)]
non_drought_vals = pivot.loc[~pivot.index.isin(drought_years)]

# Apply Welch's t-test column-wise
t_vals, p_vals = ttest_ind(
    drought_vals, non_drought_vals,
    axis=0, equal_var=False, nan_policy='omit'
)

# Convert result arrays to DataFrames with same multi-index columns
p_df = pd.DataFrame([p_vals], columns=pivot.columns).T.reset_index()
p_df.columns = ['latitude', 'longitude', 'season', 'p_value']

# Merge with original mergedfv
mergedfv = pd.merge(mergedfv, p_df, on=['latitude', 'longitude', 'season'], how='left')

############################################################################################
# Convert back to xarray
uanomalies = mergedfu.set_index(['date', 'latitude', 'longitude'])['u_anomaly'].to_xarray()
upval = mergedfu.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()

vanomalies = mergedfv.set_index(['date', 'latitude', 'longitude'])['v_anomaly'].to_xarray()
vpval = mergedfv.set_index(['date', 'latitude', 'longitude'])['p_value'].to_xarray()
uanomalies

<xarray.DataArray 'u_anomaly' (date: 137, latitude: 401, longitude: 521)>
array([[[ 1.117199  ,  1.1379781 ,  1.1627176 , ..., -0.09045076,
         -0.09492159, -0.10733461],
        [ 1.1662984 ,  1.1973476 ,  1.2200202 , ..., -0.06286812,
         -0.07071352, -0.09800816],
        [ 1.1901753 ,  1.240669  ,  1.2744527 , ..., -0.03100538,
         -0.03309917, -0.07058859],
        ...,
        [ 0.8121805 ,  0.80545855,  0.8019972 , ..., -0.7281461 ,
         -0.7277012 , -0.72562873],
        [ 0.8466423 ,  0.8312452 ,  0.8145026 , ..., -0.6244677 ,
         -0.6240391 , -0.6229812 ],
        [ 0.82237476,  0.8037224 ,  0.78356725, ..., -0.5683316 ,
         -0.5664653 , -0.56368756]],

       [[ 0.11273259,  0.13139579,  0.13497654, ...,  1.5855842 ,
          1.6273375 ,  1.6803541 ],
        [ 0.1597161 ,  0.16667685,  0.16395873, ...,  1.6782708 ,
          1.7158356 ,  1.7560868 ],
        [ 0.19628836,  0.18330552,  0.16956313, ...,  1.7737732 ,
          1.7995114 ,  1.8214293 ],
...
        [-0.1636678 , -0.1579386 , -0.15369606, ...,  0.5172349 ,
          0.52425534,  0.5298326 ],
        [-0.13061655, -0.13054061, -0.13023686, ...,  0.43405357,
          0.4403795 ,  0.44531655],
        [-0.1154474 , -0.11353767, -0.11116123, ...,  0.32553577,
          0.33096108,  0.33651662]],

       [[ 0.15513355,  0.19704986,  0.2122842 , ..., -0.73630524,
         -0.75180006, -0.76763105],
        [ 0.16564786,  0.20247519,  0.22403556, ..., -0.71827126,
         -0.7329302 , -0.74880505],
        [ 0.19520521,  0.21187186,  0.21362972, ..., -0.70644426,
         -0.72312164, -0.7355242 ],
        ...,
        [ 1.3193156 ,  1.3273233 ,  1.3338444 , ..., -1.7615005 ,
         -1.7574098 , -1.7550877 ],
        [ 1.2901921 ,  1.2811537 ,  1.2798297 , ..., -1.7398642 ,
         -1.7354913 , -1.7321819 ],
        [ 1.291364  ,  1.2613728 ,  1.2438923 , ..., -1.7237074 ,
         -1.7173057 , -1.7110991 ]]], dtype=float32)
Coordinates:
  * date       (date) object 'DJF 1991' 'DJF 1992' ... 'SON 2023' 'SON 2024'
  * latitude   (latitude) float64 -20.0 -19.75 -19.5 -19.25 ... 79.5 79.75 80.0
  * longitude  (longitude) float64 -50.0 -49.75 -49.5 -49.25 ... 79.5 79.75 80.0

In [13]:
def export_to_geotiff(path, u_data, v_data, upval, vpval, lat, lon, description):
    # Flip latitude if needed (GeoTIFF expects top-to-bottom order)
    if lat[0] > lat[-1]:
        u_data = u_data[::-1]
        v_data = v_data[::-1]
        upval = u_data[::-1]
        vpval = v_data[::-1]
        lat = lat[::-1]

    # Get transform
    pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1)
    pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1)
    transform = from_origin(lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y)

    # Stack u and v into a 3D array (bands, height, width)
    data = np.stack([u_data, v_data, upval, vpval])  # shape: (2, lat, lon)

    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=u_data.shape[0],
        width=u_data.shape[1],
        count=4,  # 2 bands: u and v
        dtype=u_data.dtype,
        crs="EPSG:4326",
        transform=transform,
        description=description
    ) as dst:
        dst.write(data[0], 1)  # Band 1 - u
        dst.write(data[1], 2) # Band 2 - v
        dst.write(data[2], 3)  # Band 3 - upval
        dst.write(data[3], 4) # Band 4 - vpval
        
# Export regression slope map
output_dir = 'Wind850'  # no leading slash unless it's an absolute path
os.makedirs(output_dir, exist_ok=True)

for d in range(137):
    file_path = os.path.join(output_dir, str(uanomalies['date'][d].values) + '.tif')
    export_to_geotiff(file_path, uanomalies[d], vanomalies[d], upval[d], vpval[d], uanomalies.latitude.values, uanomalies.longitude.values, 'Wind850 Seasonal Anomalies of '+str(uanomalies['date'][d].values))
